# SAE + CNN Integration for Mechanistic Interpretability

This notebook integrates a topK Sparse Autoencoder (SAE) into the CNN 2D classifier using PyTorch hooks for mechanistic interpretability and causal testing.

## Overview
- Load trained CNN 2D and topK SAE models
- Register hooks to intercept intermediate feature maps
- Encode/decode features through SAE to get sparse codes
- Analyze which concepts are learned and important
- Run causal intervention tests (ablation, patching)

## Section 1: Import Required Libraries and Load Models

In [16]:
import sys
import os
import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import joblib

# Add scripts directory to path
sys.path.insert(0, '/home/student/s/ssahu/share/aigm-classifier/scripts')

from train_cnn_2d import CNN2D
from train_topk_sae import TopKSAE
from sae_cnn_integration import SAECNNIntegration, CausalTestSuite
from utils import ROOT_DIR, DATA_DIR

# Set up matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {device}")
print(f"🔧 ROOT_DIR: {ROOT_DIR}")
print(f"📂 DATA_DIR: {DATA_DIR}")

🖥️  Using device: cpu
🔧 ROOT_DIR: /share/users/student/s/ssahu/aigm-classifier
📂 DATA_DIR: /share/users/student/s/ssahu/aigm-classifier/data


In [20]:
# Reload modules to get latest changes
import importlib
import sae_cnn_integration
importlib.reload(sae_cnn_integration)
from sae_cnn_integration import SAECNNIntegration, CausalTestSuite

# Define paths to trained models
cnn_model_path = os.path.join(ROOT_DIR, "models", "cnn_2d_model.pt")
sae_model_dir = os.path.join(ROOT_DIR, "models", "topk_sae")

# Check if models exist
print(f"\n📂 Checking for trained models...")
print(f"   CNN model: {cnn_model_path}")
print(f"   CNN exists: {os.path.exists(cnn_model_path)}")
print(f"   SAE model: {sae_model_dir}")
print(f"   SAE exists: {os.path.exists(sae_model_dir)}")

# Initialize the integrated model
print(f"\n🔧 Initializing SAE+CNN integration...")
integration = SAECNNIntegration(
    cnn_model_path=cnn_model_path,
    sae_model_dir=sae_model_dir,
    target_layer='conv6',
    device=device
)

print(f"\n✅ Models loaded successfully!")
print(f"   CNN parameters: {sum(p.numel() for p in integration.cnn.parameters()):,}")
print(f"   SAE input shape: {integration.sae_config['input_shape']}")
print(f"   SAE nb_concepts: {integration.sae_config['nb_concepts']}")
print(f"   SAE top_k: {integration.sae_config['top_k']}")
print(f"\n⚠️  NOTE: The SAE was trained on spectrograms, not CNN features.")
print(f"   For optimal results, train a new SAE on conv6 activations.")


📂 Checking for trained models...
   CNN model: /share/users/student/s/ssahu/aigm-classifier/models/cnn_2d_model.pt
   CNN exists: True
   SAE model: /share/users/student/s/ssahu/aigm-classifier/models/topk_sae
   SAE exists: True

🔧 Initializing SAE+CNN integration...
📂 Loading CNN model from /share/users/student/s/ssahu/aigm-classifier/models/cnn_2d_model.pt
   ✓ CNN loaded
📂 Loading SAE model from /share/users/student/s/ssahu/aigm-classifier/models/topk_sae
   ✓ SAE loaded
   ✓ Patch scaler loaded
   ✓ Hook registered on layer 'conv6'

✅ Models loaded successfully!
   CNN parameters: 1,704,322
   SAE input shape: 32768
   SAE nb_concepts: 128
   SAE top_k: 32

⚠️  NOTE: The SAE was trained on spectrograms, not CNN features.
   For optimal results, train a new SAE on conv6 activations.


## Section 2: Test Integration with Sample Input

Create synthetic test data and run forward pass through CNN with SAE hooks.

In [21]:
# Create synthetic test input (batch_size=4, channels=1, height=128, width=128)
batch_size = 4
synthetic_input = torch.randn(batch_size, 1, 128, 128).to(device)

print(f"🔍 Testing forward pass with synthetic input...")
print(f"   Input shape: {synthetic_input.shape}")

# Forward pass through integrated CNN+SAE
result = integration.forward_with_sae(synthetic_input)

print(f"\n✅ Forward pass successful!")
print(f"\n📊 Output shapes:")
print(f"   CNN output: {result['cnn_output'].shape}")
print(f"   Activations (conv6): {result['activations'].shape}")
print(f"   Sparse codes: {result['sparse_codes'].shape}")
print(f"   SAE reconstruction: {result['sae_reconstruction'].shape}")

# Compute some metrics
sparse_codes = result['sparse_codes']
print(f"\n📈 Sparse code statistics:")
print(f"   Mean activation: {sparse_codes.mean():.4f}")
print(f"   Max activation: {sparse_codes.max():.4f}")
print(f"   Sparsity (% zeros): {(sparse_codes == 0).float().mean():.2%}")
print(f"   L1 norm (avg): {sparse_codes.abs().sum(dim=1).mean():.4f}")

# Reconstruction loss
original_acts = result['activations']
reconstructed_acts = result['sae_reconstruction']
recon_loss = nn.MSELoss()(reconstructed_acts, original_acts)
print(f"   SAE Reconstruction MSE: {recon_loss:.4f}")

🔍 Testing forward pass with synthetic input...
   Input shape: torch.Size([4, 1, 128, 128])
🔧 Activation shape: (4, 4, 4, 512), flattened to SAE input: (64, 512)
   Normalized to: mean=-0.0000, std=1.0000
   ⚠️  Padding from 512 to 32768 dimensions with zeros

✅ Forward pass successful!

📊 Output shapes:
   CNN output: torch.Size([4, 2])
   Activations (conv6): torch.Size([4, 512, 4, 4])
   Sparse codes: torch.Size([64, 128])
   SAE reconstruction: torch.Size([4, 512, 4, 4])

📈 Sparse code statistics:
   Mean activation: -1.7281
   Max activation: 13.5149
   Sparsity (% zeros): 0.00%
   L1 norm (avg): 291.7336
   SAE Reconstruction MSE: 285.9397


## Section 3: Visualize SAE Activations and Reconstructions

Analyze feature maps and sparse code patterns.

In [ ]:
# Get activations from test input
print("📸 Extracting activation patterns...")
result = integration.forward_with_sae(synthetic_input)

original_activations = result['activations'].cpu().detach().numpy()
reconstructed_activations = result['sae_reconstruction'].cpu().detach().numpy()
sparse_codes = result['sparse_codes'].cpu().detach().numpy()

# Visualize activations for first sample
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('SAE Activation Analysis - First Sample', fontsize=14, fontweight='bold')

# Original activations (show first 3 channels)
for ch in range(3):
    ax = axes[0, ch]
    im = ax.imshow(original_activations[0, ch], cmap='viridis')
    ax.set_title(f'Original Conv6 Ch{ch}')
    plt.colorbar(im, ax=ax)

# Reconstructed activations
for ch in range(3):
    ax = axes[1, ch]
    im = ax.imshow(reconstructed_activations[0, ch], cmap='viridis')
    ax.set_title(f'SAE Reconstructed Ch{ch}')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print(f"✅ Visualization complete!")

In [ ]:
# Analyze sparse code patterns
print("📊 Sparse Code Pattern Analysis")
print("=" * 60)

# Get top-k concepts for each sample
result = integration.forward_with_sae(synthetic_input)
sparse_codes = result['sparse_codes'].cpu().detach().numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sparse Code Analysis', fontsize=14, fontweight='bold')

# 1. Histogram of concept activations across all samples
ax = axes[0, 0]
all_activations = sparse_codes[sparse_codes > 0]  # Non-zero activations
ax.hist(all_activations, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Activation Magnitude')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Non-Zero Activations')
ax.grid(True, alpha=0.3)

# 2. Sparsity per sample
ax = axes[0, 1]
sparsity_per_sample = (sparse_codes == 0).mean(axis=1) * 100
ax.bar(range(len(sparsity_per_sample)), sparsity_per_sample, color='coral', alpha=0.7)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Sparsity (%)')
ax.set_title('Sparsity per Sample')
ax.grid(True, alpha=0.3, axis='y')

# 3. Top concepts (most activated)
ax = axes[1, 0]
concept_strength = sparse_codes.sum(axis=0)
top_k_concepts = np.argsort(-concept_strength)[:20]
ax.barh(range(20), concept_strength[top_k_concepts], color='lightgreen', alpha=0.7)
ax.set_yticks(range(20))
ax.set_yticklabels([f'Concept {i}' for i in top_k_concepts])
ax.set_xlabel('Total Activation Strength')
ax.set_title('Top 20 Most Activated Concepts')
ax.grid(True, alpha=0.3, axis='x')

# 4. Concept activation matrix (heatmap for first sample)
ax = axes[1, 1]
sample_codes = sparse_codes[0].reshape(1, -1)
# Show only top 50 concepts for clarity
im = ax.imshow(sample_codes[:, :50], cmap='hot', aspect='auto')
ax.set_xlabel('Concept Index (top 50)')
ax.set_ylabel('Sample')
ax.set_title('Activation Heatmap - First Sample (top 50 concepts)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print(f"\n📈 Statistics:")
print(f"   Overall sparsity: {(sparse_codes == 0).mean():.2%}")
print(f"   Mean non-zero activation: {all_activations.mean():.4f}")
print(f"   Std of non-zero activations: {all_activations.std():.4f}")
print(f"   Top 5 concepts: {top_k_concepts[:5].tolist()}")

## Section 4: Causal Intervention - Ablation Tests

Test the causal effect of removing specific sparse concepts on model predictions.

In [ ]:
# Step 1: Get original predictions
print("🔬 Testing Causal Effects of Ablating Concepts")
print("=" * 60)

test_input = synthetic_input.clone()

# Original forward pass
original_result = integration.forward_with_sae(test_input)
original_logits = original_result['cnn_output']
original_pred = torch.argmax(original_logits, dim=1)
original_conf = torch.softmax(original_logits, dim=1).max(dim=1)[0]

print(f"\n📍 Original Predictions:")
print(f"   Logits shape: {original_logits.shape}")
print(f"   Predictions: {original_pred.cpu().numpy()}")
print(f"   Confidence: {original_conf.cpu().numpy()}")

# Step 2: Test ablating different numbers of concepts
result = integration.forward_with_sae(test_input)
sparse_codes = result['sparse_codes'].cpu().detach().numpy()

print(f"\n🔍 Testing ablation of top-k concepts...")

# Get top concepts for first sample
sample_codes = sparse_codes[0]
top_concepts = np.argsort(-sample_codes)

ablation_results = {
    'num_ablated': [],
    'pred_change': [],
    'conf_change': [],
    'logit_diff': []
}

# Test ablating increasing numbers of top concepts
for k in [1, 5, 10, 20, 50]:
    if k <= len(top_concepts):
        # Ablate top-k concepts
        concepts_to_ablate = top_concepts[:k].tolist()
        
        ablated_result = integration.causal_intervention_ablate(
            test_input, 
            concept_indices=concepts_to_ablate
        )
        
        # Note: This is simplified - in practice you need to properly forward through the rest of the CNN
        # For now, we just analyze the sparse codes
        codes_original = ablated_result['sparse_codes']
        codes_ablated = ablated_result['sparse_codes_ablated']
        
        code_norm_change = (codes_ablated - codes_original).abs().mean()
        
        ablation_results['num_ablated'].append(k)
        ablation_results['logit_diff'].append(float(code_norm_change.cpu().numpy()))
        
        print(f"   ✓ Ablated top-{k} concepts | Code change: {code_norm_change:.4f}")

# Plot ablation results
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ablation_results['num_ablated'], ablation_results['logit_diff'], 
        marker='o', linewidth=2, markersize=8, color='darkred')
ax.set_xlabel('Number of Top Concepts Ablated', fontsize=12)
ax.set_ylabel('Sparse Code Change', fontsize=12)
ax.set_title('Causal Effect: Concept Ablation', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Ablation tests complete!")

## Section 5: Concept Importance Analysis

Extract sparse codes from a dataset and analyze which concepts are most important.

In [ ]:
# Analyze concept importance using synthetic data batch
print("📊 Concept Importance Analysis")
print("=" * 60)

# Create a batch of data
batch_size = 32
test_batch = torch.randn(batch_size, 1, 128, 128).to(device)

# Extract sparse codes
concept_activations = integration.get_concept_activations(
    dataloader=[(test_batch, torch.zeros(batch_size))],
    n_batches=1
)

print(f"✓ Extracted concept activations shape: {concept_activations.shape}")

# Analyze importance
importance_stats = integration.analyze_concept_importance(concept_activations)

# Get top 20 concepts by mean activation
concept_means = {
    int(k.split('_')[1]): v['mean_activation'] 
    for k, v in importance_stats.items()
}
top_20_concepts = sorted(concept_means.items(), key=lambda x: abs(x[1]), reverse=True)[:20]

print(f"\n🎯 Top 20 Concepts by Mean Activation:")
for rank, (concept_idx, mean_val) in enumerate(top_20_concepts, 1):
    stats = importance_stats[f'concept_{concept_idx}']
    print(f"   {rank:2d}. Concept {concept_idx:3d}: mean={mean_val:7.4f}, "
          f"std={stats['std_activation']:7.4f}, sparsity={stats['sparsity']:.2%}")

# Visualize concept importance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Mean activations of top 30 concepts
ax = axes[0]
top_30_idxs = [c[0] for c in sorted(concept_means.items(), 
                                     key=lambda x: abs(x[1]), 
                                     reverse=True)[:30]]
top_30_means = [concept_means[i] for i in top_30_idxs]
ax.barh(range(len(top_30_idxs)), top_30_means, color='skyblue', alpha=0.7)
ax.set_yticks(range(len(top_30_idxs)))
ax.set_yticklabels([f'C{i}' for i in top_30_idxs], fontsize=8)
ax.set_xlabel('Mean Activation')
ax.set_title('Top 30 Concepts by Mean Activation')
ax.grid(True, alpha=0.3, axis='x')

# Sparsity distribution
ax = axes[1]
sparsities = [importance_stats[f'concept_{i}']['sparsity'] for i in range(len(importance_stats))]
ax.hist(sparsities, bins=30, color='lightcoral', alpha=0.7, edgecolor='black')
ax.set_xlabel('Sparsity (fraction of zeros)')
ax.set_ylabel('Number of Concepts')
ax.set_title('Distribution of Sparsity Across Concepts')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n✅ Concept importance analysis complete!")

## Section 6: Save Integration Configuration

Save the integration setup for reproducibility and future causal testing.

In [ ]:
# Save integration configuration
print("💾 Saving Integration Configuration")
print("=" * 60)

output_dir = os.path.join(ROOT_DIR, "models", "sae_cnn_integrated")
integration.save_integration_config(output_dir)

# Create a detailed report
report = {
    "integration_setup": {
        "cnn_model_path": str(cnn_model_path),
        "sae_model_dir": str(sae_model_dir),
        "target_layer": "conv6",
        "device": str(device),
        "timestamp": str(np.datetime64('today'))
    },
    "model_info": {
        "cnn_parameters": int(sum(p.numel() for p in integration.cnn.parameters())),
        "sae_input_shape": integration.sae_config['input_shape'],
        "sae_nb_concepts": integration.sae_config['nb_concepts'],
        "sae_top_k": integration.sae_config['top_k'],
    },
    "metrics": {
        "activation_shape": str(result['activations'].shape),
        "sparse_codes_shape": str(result['sparse_codes'].shape),
        "sparsity": float((sparse_codes == 0).mean()),
        "mean_activation": float(sparse_codes[sparse_codes > 0].mean()),
        "reconstruction_mse": float(nn.MSELoss()(reconstructed_acts, original_acts))
    },
    "usage_instructions": {
        "forward_pass": "result = integration.forward_with_sae(input_tensor)",
        "get_sparse_codes": "codes = result['sparse_codes']",
        "ablate_concepts": "ablated = integration.causal_intervention_ablate(input_tensor, concept_indices=[1, 5, 10])",
        "analyze_importance": "importance = integration.analyze_concept_importance(concept_activations)"
    }
}

report_path = os.path.join(output_dir, "integration_report.json")
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Configuration saved to: {output_dir}/")
print(f"   - integration_config.json")
print(f"   - integration_report.json")

# Display summary
print(f"\n📋 Integration Summary:")
print(f"   CNN parameters: {report['model_info']['cnn_parameters']:,}")
print(f"   SAE concepts: {report['model_info']['sae_nb_concepts']}")
print(f"   SAE top-k: {report['model_info']['sae_top_k']}")
print(f"   Overall sparsity: {report['metrics']['sparsity']:.2%}")
print(f"   Reconstruction MSE: {report['metrics']['reconstruction_mse']:.4f}")

print(f"\n🎯 Next Steps:")
print(f"   1. Load real test data instead of synthetic input")
print(f"   2. Implement full causal intervention through entire CNN")
print(f"   3. Run systematic ablation studies on different concept subsets")
print(f"   4. Analyze concept-to-class relationships")
print(f"   5. Create concept visualization/interpretation tools")

## Advanced: Full Causal Testing Pipeline

For more sophisticated mechanistic interpretability research, implement proper layer-level interventions.

In [ ]:
# Example: Concept-specific analysis
print("🔬 Advanced Mechanistic Interpretability Analysis")
print("=" * 60)

# Get sparse codes for a batch
test_batch = torch.randn(8, 1, 128, 128).to(device)
result = integration.forward_with_sae(test_batch)
sparse_codes = result['sparse_codes'].cpu().detach().numpy()

# Find concepts that are highly selective (activate in some samples but not others)
mean_activation = sparse_codes.mean(axis=0)
std_activation = sparse_codes.std(axis=0)

# Selectivity = std / (std + mean) - higher means more selective
selectivity = std_activation / (std_activation + mean_activation + 1e-7)
top_selective_concepts = np.argsort(-selectivity)[:10]

print(f"\n🎯 Top 10 Most Selective Concepts:")
print(f"   (concepts that activate in some samples but not others)")
for rank, concept_idx in enumerate(top_selective_concepts, 1):
    print(f"   {rank:2d}. Concept {concept_idx:3d}: "
          f"mean={mean_activation[concept_idx]:7.4f}, "
          f"selectivity={selectivity[concept_idx]:7.4f}")

# Analyze which concepts activate together
print(f"\n📊 Concept Co-activation Analysis:")
# Compute pairwise correlations for top 20 concepts
top_concepts = np.argsort(-mean_activation)[:20]
concept_subset = sparse_codes[:, top_concepts]
correlation_matrix = np.corrcoef(concept_subset.T)

# Find strongly correlated pairs
strong_pairs = []
for i in range(len(top_concepts)):
    for j in range(i+1, len(top_concepts)):
        corr = correlation_matrix[i, j]
        if abs(corr) > 0.5:
            strong_pairs.append((top_concepts[i], top_concepts[j], corr))

strong_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for rank, (c1, c2, corr) in enumerate(strong_pairs[:5], 1):
    print(f"   {rank}. Concept {c1} <-> Concept {c2}: correlation={corr:.3f}")

print(f"\n✅ Advanced analysis complete!")
print(f"\n💡 These selective and co-activated concepts are good candidates for interpretation!")
print(f"   Next: Try to understand what visual features they correspond to.")